In [ ]:
from sqlalchemy import create_engine
import pandas as pd
from queries import get_rentals_month_query

In [ ]:
def rentals_month(connection ,month, year):
    query = get_rentals_month_query()
    df = pd.read_sql(query, connection, params=(month, year))
    return df

In [ ]:
def rental_count_month(df,month,year):
    col_name = f"rentals_{month:02d}_{year}"
    rental_counts = df.groupby('customer_id')['rental_id'].count().reset_index()
    rental_counts.columns = ['customer_id', col_name]
    return rental_counts

In [ ]:
def compare_rentals(df1,df2):
    merged_df = pd.merge(df1, df2, on='customer_id', how='outer')
    count_columns = [col for col in merged_df.columns if col != 'customer_id']
    if len(count_columns) == 2:
        col_m1, col_m2 = count_columns
        merged_df['difference'] = merged_df[col_m2] - merged_df[col_m1]
    return merged_df

In [ ]:
if __name__ == "__main__":
    try:
        USER = 'root'
        PASSWORD = '35459583'
        HOST = 'localhost'
        PORT = '3306'
        DATABASE = 'sakila'
        sql_string = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
        connection = create_engine(sql_string)
        df_rentals_Aug = rentals_month(connection, 8, 2005)
        df_rentals_Jul = rentals_month(connection, 6, 2005)
        if df_rentals_Jul is not None and df_rentals_Aug is not None:
            customer_rental_count_Aug = rental_count_month(df_rentals_Aug,8,2005)
            customer_rental_count_Jul = rental_count_month(df_rentals_Jul,6,2005)
            merged_df= compare_rentals(customer_rental_count_Jul,customer_rental_count_Aug)
            display(merged_df)
        else:
            print("No data found")
    except Exception as e:
        print(e)
